In [ ]:
# ============================================
# DFD01 T=32 — RESUME PREPROCESSING
# ============================================
# Resumes DFD01 T=32 preprocessing from partial data.
# Input datasets:
#   1. "dfd01-preprocessed-t32-partial" — already processed 2973 fake videos
#   2. "dfdc-dataset-02-dfd-01" — raw videos (only DFD01 is used)
#
# This notebook:
#   1. Extracts partial tar.gz and copies already-processed face crops
#   2. Processes ONLY remaining unprocessed videos
#   3. Packages everything into preprocessed_DFD01_32.tar.gz
#
# Estimated time: ~2 hours (only ~460 remaining videos)

NUM_FRAMES = 32
OUTPUT_SIZE = 224
MIN_FACE_CONF = 0.90
MIN_DETECTION_RATIO = 0.55
DETECTOR_MAX_SIDE = 720  # Optimized for speed

import subprocess, sys, os, time, shutil, json
import glob as glob_mod

# Step 1: Install dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch',
                'opencv-python-headless', 'tqdm'],
               check=True)
print('Dependencies installed.')

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 3: Extract and copy partial preprocessed data
OUTPUT_DIR = '/kaggle/working/preprocessed_DFD01_32'
error_new = 0  # Initialize to avoid NameError if all videos already processed

# Kaggle stores uploaded tar.gz as-is, so we need to extract first
partial_input = '/kaggle/input/dfd01-preprocessed-t32-partial'
tar_files = glob_mod.glob(os.path.join(partial_input, '*.tar.gz'))
if tar_files:
    print(f'Found tar.gz in partial dataset: {tar_files[0]}')
    print('Extracting...')
    t0 = time.time()
    subprocess.run(['tar', 'xzf', tar_files[0], '-C', '/kaggle/working'], check=True)
    elapsed = time.time() - t0
    print(f'Extracted in {elapsed:.1f}s')

# Find partial data (either extracted from tar.gz or already as folders)
partial_candidates = [
    OUTPUT_DIR,  # tar.gz extracts here
    os.path.join(partial_input, 'preprocessed_DFD01_32'),
    partial_input,
]

partial_found = False
for c in partial_candidates:
    if os.path.exists(c) and (os.path.isdir(os.path.join(c, 'fake')) or os.path.isdir(os.path.join(c, 'real'))):
        # If it's not OUTPUT_DIR, copy it there
        if os.path.abspath(c) != os.path.abspath(OUTPUT_DIR):
            print(f'Copying partial data from {c} to {OUTPUT_DIR}...')
            if os.path.exists(OUTPUT_DIR):
                shutil.rmtree(OUTPUT_DIR)
            shutil.copytree(c, OUTPUT_DIR)
        partial_found = True
        break

if not partial_found:
    print('WARNING: Partial dataset not found! Will process ALL videos from scratch.')
    print('Listing /kaggle/input/:')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            indent = '  ' * level
            print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')

os.makedirs(os.path.join(OUTPUT_DIR, 'real'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'fake'), exist_ok=True)

# Count existing
existing_real = len([d for d in os.listdir(os.path.join(OUTPUT_DIR, 'real'))
                     if os.path.isdir(os.path.join(OUTPUT_DIR, 'real', d))])
existing_fake = len([d for d in os.listdir(os.path.join(OUTPUT_DIR, 'fake'))
                     if os.path.isdir(os.path.join(OUTPUT_DIR, 'fake', d))])
print(f'Partial data ready: {existing_real} real, {existing_fake} fake')

# Step 4: Find raw DFD01 videos
print('\n=== Finding raw DFD01 videos ===')
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v'}

def find_raw_datasets():
    datasets = []
    for root, dirs, files in os.walk('/kaggle/input'):
        # Skip the partial preprocessed dataset (it has no raw videos)
        if 'preprocessed' in root.lower() or 'partial' in root.lower():
            continue
        label_dirs = [d for d in dirs if d.lower() in ('real', 'fake', 'original', 'manipulated', 'actors', 'altered', 'deepfake', 'df')]
        if not label_dirs:
            continue
        has_videos = False
        video_count = 0
        for ld in label_dirs:
            subdir = os.path.join(root, ld)
            if os.path.isdir(subdir):
                for sr, sd, sf in os.walk(subdir):
                    vc = sum(1 for f in sf if os.path.splitext(f)[1].lower() in VIDEO_EXTS)
                    video_count += vc
                    if vc > 0:
                        has_videos = True
        if has_videos:
            datasets.append((root, video_count))
            print(f'  Found: {root} ({video_count} videos)')
    return datasets

raw_datasets = find_raw_datasets()

# Find DFD01
dfd01_path = None
for path, count in raw_datasets:
    path_lower = path.lower()
    if 'dfd' in path_lower and 'dfdc' not in path_lower:
        dfd01_path = path
        print(f'\nIdentified DFD01: {path} ({count} videos)')
        break

if dfd01_path is None:
    raise RuntimeError('DFD01 raw dataset not found! Make sure dfdc-dataset-02-dfd-01 is added as input.')

# Step 5: Find which videos are already processed
existing_ids = set()
for label_dir in ['real', 'fake']:
    label_path = os.path.join(OUTPUT_DIR, label_dir)
    if os.path.isdir(label_path):
        for vid_dir in os.listdir(label_path):
            if os.path.isdir(os.path.join(label_path, vid_dir)):
                existing_ids.add(vid_dir)

print(f'\nAlready processed video IDs: {len(existing_ids)}')

# Step 6: Process remaining videos using project's preprocess code
sys.path.insert(0, '/kaggle/working/project')
from preprocess_videos import (
    PreprocessConfig, find_videos, infer_label_from_path, make_safe_video_id,
    process_video, select_device, build_summary, save_manifest_csv
)
from facenet_pytorch import MTCNN
from pathlib import Path
from tqdm import tqdm
import torch

input_root = Path(dfd01_path)
all_videos = find_videos(str(input_root))
print(f'Total raw DFD01 videos: {len(all_videos)}')

# Filter: only process videos whose ID is NOT in existing_ids
to_process = []
already_done = []
for vp in all_videos:
    vid_id = make_safe_video_id(vp, input_root)
    if vid_id in existing_ids:
        already_done.append(vp)
    else:
        to_process.append(vp)

print(f'Already processed: {len(already_done)}')
print(f'Remaining to process: {len(to_process)}')

if len(to_process) == 0:
    print('All videos already processed!')
else:
    # Auto-compute min_saved_faces as 75% of max_frames
    min_saved = max(1, int(NUM_FRAMES * 0.75))
    
    cfg = PreprocessConfig(
        input_root=dfd01_path,
        output_root=OUTPUT_DIR,
        max_frames=NUM_FRAMES,
        output_size=OUTPUT_SIZE,
        min_face_confidence=MIN_FACE_CONF,
        min_detection_ratio=MIN_DETECTION_RATIO,
        min_saved_faces=min_saved,
        detector_max_side=DETECTOR_MAX_SIDE,
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )
    cfg.validate()
    
    device = select_device(cfg.device)
    print(f'Device: {device}')
    
    detector = MTCNN(
        image_size=cfg.output_size,
        margin=0,
        min_face_size=20,
        keep_all=True,
        post_process=False,
        device=device,
    )
    
    rows = []
    t0 = time.time()
    
    for video_path in tqdm(to_process, desc='Processing remaining videos'):
        try:
            row = process_video(video_path, detector, cfg, input_root=input_root)
            rows.append(row)
        except Exception as e:
            import traceback
            rows.append({
                'status': 'error',
                'reason': str(e),
                'label': '',
                'video_path': str(video_path),
                'traceback': traceback.format_exc(limit=1),
            })
    
    elapsed = (time.time() - t0) / 60
    saved_new = sum(1 for r in rows if r.get('status') == 'saved')
    dropped_new = sum(1 for r in rows if r.get('status') == 'dropped')
    error_new = sum(1 for r in rows if r.get('status') == 'error')
    print(f'\nRemaining videos processed in {elapsed:.1f} min')
    print(f'  New saved: {saved_new}, dropped: {dropped_new}, errors: {error_new}')

# Step 7: Build full summary
print('\n=== Building full summary ===')

final_real = len([d for d in os.listdir(os.path.join(OUTPUT_DIR, 'real'))
                  if os.path.isdir(os.path.join(OUTPUT_DIR, 'real', d))])
final_fake = len([d for d in os.listdir(os.path.join(OUTPUT_DIR, 'fake'))
                  if os.path.isdir(os.path.join(OUTPUT_DIR, 'fake', d))])

print(f'Final count: {final_real} real + {final_fake} fake = {final_real + final_fake} total')

summary = {
    'total_videos': len(all_videos),
    'saved_videos': final_real + final_fake,
    'real_saved': final_real,
    'fake_saved': final_fake,
    'dropped_videos': len(all_videos) - (final_real + final_fake),
    'error_videos': error_new,
    'config': {
        'max_frames': NUM_FRAMES,
        'output_size': OUTPUT_SIZE,
        'min_face_confidence': MIN_FACE_CONF,
        'min_detection_ratio': MIN_DETECTION_RATIO,
        'detector_max_side': DETECTOR_MAX_SIDE,
    }
}
with open(os.path.join(OUTPUT_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary saved.')

# Step 8: Package
print('\nPackaging preprocessed_DFD01_32.tar.gz...')
os.system('tar czf /kaggle/working/preprocessed_DFD01_32.tar.gz -C /kaggle/working preprocessed_DFD01_32/')
tar_size = os.path.getsize('/kaggle/working/preprocessed_DFD01_32.tar.gz') / (1024**2)
print(f'preprocessed_DFD01_32.tar.gz: {tar_size:.1f} MB')

print(f'\n{"="*60}')
print('DFD01 T=32 PREPROCESSING COMPLETE')
print(f'{"="*60}')
print(f'Output: preprocessed_DFD01_32.tar.gz ({tar_size:.1f} MB)')
print(f'Real: {final_real}, Fake: {final_fake}')
print('Upload this as Kaggle dataset for multi-dataset training.')